# 04. GNN Models

This notebook builds a single-node classification pipeline on the `10k` sampled graph.

Supported models:

- `graphsage`
- `gat`
- `ggnn`

Recommended first run on CPU:

- `MODEL_NAME = "graphsage"`
- `FEATURE_GROUP = "eth_twitter_combined_features"`
- `EPOCHS = 80`


In [ ]:
from pathlib import Path
import copy
import random

import dgl
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dgl.nn import GATConv, GatedGraphConv, SAGEConv
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
nodes_df = pd.read_csv(ROOT / "wash_trading_gnn_nodes_10000.csv")
edges_df = pd.read_csv(ROOT / "wash_trading_gnn_edges_10000.csv")


In [ ]:
MODEL_NAME = "graphsage"  # one of: graphsage, gat, ggnn
FEATURE_GROUP = "eth_twitter_combined_features"
ADD_GRAPH_STATS = True
RANDOM_STATE = 42
HIDDEN_DIM = 64
DROPOUT = 0.2
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 80
PATIENCE = 15

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


In [ ]:
feature_groups = {
    "features": [c for c in nodes_df.columns if c.startswith("features_")],
    "normalized_log_features": [c for c in nodes_df.columns if c.startswith("normalized_log_features_")],
    "twitter_combined_features": [c for c in nodes_df.columns if c.startswith("twitter_combined_features_")],
    "eth_twitter_combined_features": [c for c in nodes_df.columns if c.startswith("eth_twitter_combined_features_")],
}
graph_stat_cols = [
    "full_in_degree",
    "full_out_degree",
    "full_total_degree",
    "full_positive_touch_count",
    "full_has_self_loop",
    "sub_in_degree",
    "sub_out_degree",
    "sub_total_degree",
]

feature_cols = feature_groups[FEATURE_GROUP] + (graph_stat_cols if ADD_GRAPH_STATS else [])
print("Number of input features:", len(feature_cols))


In [ ]:
node_ids = nodes_df["node_id"].tolist()
node_to_idx = {node_id: idx for idx, node_id in enumerate(node_ids)}

src = edges_df["src_node_id"].map(node_to_idx).to_numpy()
dst = edges_df["dst_node_id"].map(node_to_idx).to_numpy()

graph = dgl.graph((src, dst), num_nodes=len(nodes_df))
graph = dgl.add_self_loop(graph)

x = torch.tensor(nodes_df[feature_cols].fillna(0.0).to_numpy(), dtype=torch.float32)
y = torch.tensor(nodes_df["label"].to_numpy(), dtype=torch.long)

train_idx, temp_idx = train_test_split(
    np.arange(len(nodes_df)),
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=nodes_df["label"],
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=nodes_df.iloc[temp_idx]["label"],
)

train_mask = torch.zeros(len(nodes_df), dtype=torch.bool)
val_mask = torch.zeros(len(nodes_df), dtype=torch.bool)
test_mask = torch.zeros(len(nodes_df), dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

train_mean = x[train_mask].mean(0, keepdim=True)
train_std = x[train_mask].std(0, keepdim=True).clamp_min(1e-6)
x = (x - train_mean) / train_std

class_counts = torch.bincount(y[train_mask])
class_weights = class_counts.sum() / (len(class_counts) * class_counts.float())
class_weights


In [ ]:
class GraphSAGEModel(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim, "mean")
        self.conv2 = SAGEConv(hidden_dim, hidden_dim, "mean")
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, out_dim)

    def forward(self, g, features):
        h = self.conv1(g, features)
        h = F.relu(h)
        h = self.dropout(h)
        h = self.conv2(g, h)
        h = F.relu(h)
        h = self.dropout(h)
        return self.classifier(h)


class GATModel(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout, num_heads=4):
        super().__init__()
        self.gat1 = GATConv(in_dim, hidden_dim, num_heads=num_heads, feat_drop=dropout, attn_drop=dropout)
        self.gat2 = GATConv(hidden_dim * num_heads, hidden_dim, num_heads=1, feat_drop=dropout, attn_drop=dropout)
        self.classifier = nn.Linear(hidden_dim, out_dim)

    def forward(self, g, features):
        h = self.gat1(g, features).flatten(1)
        h = F.elu(h)
        h = self.gat2(g, h).squeeze(1)
        h = F.elu(h)
        return self.classifier(h)


class GGNNModel(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, n_steps=3):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.ggnn = GatedGraphConv(hidden_dim, hidden_dim, n_steps=n_steps, n_etypes=1)
        self.classifier = nn.Linear(hidden_dim, out_dim)

    def forward(self, g, features):
        h = self.input_proj(features)
        etypes = torch.zeros(g.num_edges(), dtype=torch.long)
        h = self.ggnn(g, h, etypes)
        h = F.relu(h)
        return self.classifier(h)


if MODEL_NAME == "graphsage":
    model = GraphSAGEModel(x.shape[1], HIDDEN_DIM, 2, DROPOUT)
elif MODEL_NAME == "gat":
    model = GATModel(x.shape[1], HIDDEN_DIM, 2, DROPOUT)
elif MODEL_NAME == "ggnn":
    model = GGNNModel(x.shape[1], HIDDEN_DIM, 2)
else:
    raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}")

model


In [ ]:
def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 37)
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_f1:
            best_threshold = float(threshold)
            best_f1 = float(score)
    return best_threshold


def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": threshold,
        "PR-AUC": average_precision_score(y_true, y_prob),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Balanced-Accuracy": balanced_accuracy_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
    }


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(weight=class_weights)

best_state = None
best_val_pr_auc = -1.0
patience_left = PATIENCE
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    logits = model(graph, x)
    loss = criterion(logits[train_mask], y[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(graph, x)
        val_prob = torch.softmax(logits[val_mask], dim=1)[:, 1].cpu().numpy()
        val_true = y[val_mask].cpu().numpy()
        val_pr_auc = average_precision_score(val_true, val_prob)
        val_threshold = find_best_threshold(val_true, val_prob)
        val_metrics = compute_metrics(val_true, val_prob, val_threshold)

    history.append({"epoch": epoch, "train_loss": float(loss.item()), **{f"val_{k}": v for k, v in val_metrics.items()}})

    if val_pr_auc > best_val_pr_auc:
        best_val_pr_auc = float(val_pr_auc)
        best_state = copy.deepcopy(model.state_dict())
        patience_left = PATIENCE
    else:
        patience_left -= 1

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | train_loss={loss.item():.4f} | val_pr_auc={val_pr_auc:.4f} | val_f1={val_metrics['F1']:.4f}")

    if patience_left == 0:
        print(f"Early stopping at epoch {epoch}")
        break


In [ ]:
history_df = pd.DataFrame(history)
display(history_df.tail())


In [ ]:
model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    logits = model(graph, x)
    val_prob = torch.softmax(logits[val_mask], dim=1)[:, 1].cpu().numpy()
    val_true = y[val_mask].cpu().numpy()
    best_threshold = find_best_threshold(val_true, val_prob)

    test_prob = torch.softmax(logits[test_mask], dim=1)[:, 1].cpu().numpy()
    test_true = y[test_mask].cpu().numpy()
    test_metrics = compute_metrics(test_true, test_prob, best_threshold)

print(f"Model: {MODEL_NAME}")
pd.Series(test_metrics)


## Suggested usage

1. Run `graphsage` first.
2. Re-run with `gat`.
3. Re-run with `ggnn`.
4. Compare all three against notebook 2 and notebook 3 using `PR-AUC`, `F1`, and `Recall`.
